# r4pm - Process Mining Bindings

This notebook demonstrates the key features of r4pm:
- Auto-generated bindings with full IDE support
- Automatic type conversion (no manual conversion needed!)
- Registry-based data management
- Fast XES/OCEL import/export

## 1. Basic Example: Automatic Type Conversion

The key feature - pass any data type to functions, conversion happens automatically!

In [1]:
import r4pm.bindings as bindings
import r4pm

# Load an OCEL file - returns a registry ID
ocel_id = r4pm.import_item('SlimLinkedOCEL', 'test_data/order-management.xml')
print(f"✓ Loaded OCEL: {ocel_id[:20]}...")

# Use it directly
num = bindings.num_events(ocel=ocel_id)
print(f"✅ Number of events: {num}")

# Check registry - now has 2 items (original + converted)
items = r4pm.list_items()
print(f"\n📋 Registry now has {len(items)} items:")
for item in items:
    print(f"   {item['type']}: {item['id'][:30]}...")

✓ Loaded OCEL: ffd64a3d-3ffb-4592-8...
✅ Number of events: 21008

📋 Registry now has 1 items:
   SlimLinkedOCEL: ffd64a3d-3ffb-4592-8c1a-6b3d36...


In [2]:
r4pm.bindings.analysis.object_centric.get_object_attribute_changes(ocel_id,"Echo")

{'traces': {'price': [{'time': '2023-04-03T01:00:00+02:00', 'value': 99.99},
   {'time': '2023-06-02T11:39:36+02:00', 'value': 105.7},
   {'time': '2023-08-01T14:17:59+02:00', 'value': 108.19},
   {'time': '2023-09-30T16:24:43+02:00', 'value': 109.76},
   {'time': '2023-12-01T10:01:30+01:00', 'value': 112.39},
   {'time': '2024-01-30T11:45:23+01:00', 'value': 117.85}],
  'weight': [{'time': '2023-04-03T01:00:00+02:00', 'value': 0.78}]}}

In [3]:
net = r4pm.petri_net.import_pnml("test_data/sepsis-DISCovered.apnml")
x = r4pm.bindings.align_empty_trace(net)

In [4]:
r4pm.petri_net.export_pnml(net,"test_data/exported.apnml")

In [5]:
item_log = r4pm.bindings.flatten_ocel_on(ocel_id,"items")
# Create a dotted chart based o nthe flattened item log
item_log_dotted = r4pm.bindings.analysis.case_centric.get_dotted_chart(item_log)
# See how many dots there are
print(len(item_log_dotted['y_values']))

7659


## 2. Process Discovery

All discovery algorithms work with automatic conversion

In [6]:
# DFG from OCEL (auto-converts to SlimLinkedOCEL)
dfg = bindings.discover_dfg_from_ocel(ocel_id)
object_types = list(dfg['object_type_to_dfg'].keys())
print(f"✓ DFG for {len(object_types)} object types: {', '.join(object_types)}")

# OC-Declare
options = {
    "noise_threshold": 0.0,
    "o2o_mode": "None",
    "counts_for_generation": [1, 20],
    "counts_for_filter": [1, 20],
    "reduction": "None",
    "refinement": False,
    "considered_arrow_types": ["AS"]
}
constraints = bindings.discover_oc_declare(ocel_id, options)
print(f"✓ Discovered {len(constraints)} OC-Declare constraints")

✓ DFG for 6 object types: products, customers, employees, orders, items, packages
✓ Discovered 72 OC-Declare constraints


## 3. Registry Operations

Manual conversion and export when needed

In [7]:
# Get as DataFrames
ocel_dfs = r4pm.item_to_df(ocel_id)
print(f"✓ DataFrames: {list(ocel_dfs.keys())}")
print(f"  Events: {ocel_dfs['events'].shape}")
print(f"  Objects: {ocel_dfs['objects'].shape}")

# Export to file
r4pm.export_item(ocel_id, "test_data/output.xml")
print(f"\n✓ Exported OCEL XML")
r4pm.export_item(ocel_id, "test_data/output.json")
print(f"\n✓ Exported OCEL JSON")

# Clean up
for item in r4pm.list_items():
    r4pm.remove_item(item['id'])
print("✓ Registry cleaned")

✓ DataFrames: ['objects', 'relations', 'events', 'object_changes', 'o2o']
  Events: (21008, 3)
  Objects: (10840, 5)

✓ Exported OCEL XML

✓ Exported OCEL JSON
✓ Registry cleaned


## 4. Simple Import/Export API - Direct DataFrame Import

For direct DataFrame operations without registry

In [8]:
# XES import - returns (DataFrame, attributes_json)
xes, attrs = r4pm.df.import_xes("test_data/Sepsis Cases - Event Log.xes.gz")
print(f"✓ XES shape: {xes.shape}")

# OCEL import - returns dict of DataFrames
ocel = r4pm.df.import_ocel_xml("test_data/order-management.xml")
print(f"✓ OCEL events: {ocel['events'].shape}")

# Export
r4pm.df.export_xes(xes, "test_data/output.xes")
print("✓ Exported XES")

✓ XES shape: (15214, 32)
✓ OCEL events: (21008, 3)
✓ Exported XES


In [9]:
r4pm.bindings.test_some_inputs("Hello",1,3,3.333,True)

's=Hello,n=1,i=3,f=3.333,b=true'

In [10]:
for f in r4pm.list_bindings():
    print(f['name'])

oc_declare_conformance
get_object_attribute_changes
add_init_exit_events_to_ocel
log_to_activity_projection
get_variants
get_num_cases
get_num_variants
get_top_n_variants
get_projection_activities
get_dfg_of_object_type
flatten_ocel_on
discover_alpha+++
slim_link_ocel
index_link_ocel
ocel_type_stats
test_some_inputs
num_events
num_objects
discover_oc_declare
compute_fitness
align_trace
align_empty_trace
align_variants
get_event_type_of_id
locel_get_ev_id
locel_get_ev_types
locel_get_obs_of_type
get_o2o_ids
locel_get_o2o
locel_get_e2o
locel_delete_o2o
locel_get_ob_type
locel_get_ob_id
locel_get_o2o_rev
locel_add_event_type
get_e2o_ids
locel_add_object
locel_add_e2o
locel_get_evs_of_type
locel_get_ev_type
locel_get_ob_attr_vals
locel_get_ob_types
get_e2o_rev_ids
locel_get_ev_type_of
locel_construct_ocel
locel_delete_e2o
get_event_ids_of_type
locel_new
locel_get_ev_attr_val
locel_get_ob_type_of
locel_get_ob_by_id
locel_get_ev_by_id
get_object_ids_of_type
locel_get_full_ev
locel_add_event


## 5. Petri Net Alignments (PM4Py discovery + Rust alignment)

Discover a Petri net with PM4Py, convert it to an r4pm Petri net dict with
`r4pm.petri_net.from_pm4py` (no manual PNML round-trip needed), then align the
log's variants and compute fitness with the fast Rust alignment binding.

`r4pm.petri_net` also offers `import_pnml` / `export_pnml` (PNML files) and
`to_pm4py` (back to a PM4Py net).

In [11]:
from r4pm import petri_net
import pm4py

LOG = "test_data/Sepsis Cases - Event Log.xes.gz"

# 1. Discover a Petri net with PM4Py (Inductive Miner infrequent, 0.2 noise threshold)
log_pd = pm4py.read_xes(LOG)
pn, im, fm = pm4py.discover_petri_net_inductive(log_pd, noise_threshold=0.2)

# 2. Convert the PM4Py net (+ markings) to an r4pm Petri net dict
rnet = petri_net.from_pm4py(pn, im, fm)
# petri_net.export_pnml(rnet, "sepsis.pnml")   # optionally write a PNML file

# 3. Load the log into the registry
log_id = r4pm.import_item("EventLog", LOG)

# 4. Align all variants with the Rust binding and compute fitness
#    (the EventLog id is auto-projected to activity variants)
align_res = r4pm.bindings.align_variants(rnet, log_id)
fitness = r4pm.bindings.compute_fitness(align_res, rnet)
print(f"Aligned {len(align_res)} variants")
fitness

/home/aarkue/doc/projects/rustxes/.venv/lib/python3.14/site-packages/pm4py/utils.py:1005: UserWarning: In the current version, the import/export operation uses `r4pm` by default for importing/exporting files faster.
  warnings.warn(


Aligned 846 variants


{'log_fitness': 0.9693045878795846,
 'average_fitness': 0.9340322560501673,
 'perfectly_fitting_frac': 0.6666666666666666,
 'total_costs': 467}